# Anisotropic voxels and crop faces

Two things a real microscope stack brings that a simulated mask does not:

1. **voxels longer along z** than in x and y: give the voxel size, and `construct_mesh_in_physical_units` resamples
   the mask to cubic voxels before meshing and returns the mesh in physical units;
2. **cells cut by the image box**: the triangles that close them on the box are **crop faces**, not contact with the
   medium, and the cells they touch are **cut cells**.

The mask here is generated in the notebook (Voronoi cells in a ball), so it runs anywhere. Install with
`pip install "delaunay-watershed-3d[viewing]"` (matplotlib, for the figure; Polyscope, for the last cell).

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

from dw3d import get_default_mesh_reconstruction_algorithm, resample_to_isotropic
from dw3d.io import load_rec_appendix, load_rec_with_crop_faces


def generated_foam(n: int = 96, n_cells: int = 12, seed: int = 7) -> np.ndarray:
    """Voronoi cells filling a ball, on an n^3 grid of cubic voxels; label 0 is the medium."""
    rng = np.random.default_rng(seed)
    radius, centre = 0.46 * n, (n - 1) / 2
    seeds = []
    while len(seeds) < n_cells:
        point = rng.uniform(-0.75 * radius, 0.75 * radius, 3)
        if np.linalg.norm(point) < 0.75 * radius and all(np.linalg.norm(point - s) > 0.35 * radius for s in seeds):
            seeds.append(point)
    grid = np.indices((n, n, n)).reshape(3, -1).T.astype(float)
    _, nearest = cKDTree(np.asarray(seeds) + centre).query(grid)
    inside = np.linalg.norm(grid - centre, axis=1) <= radius
    return np.where(inside, nearest + 1, 0).reshape(n, n, n).astype(np.uint16)


foam = generated_foam()     # cubic voxels of 0.5 um
stack = foam[::2]           # every second z slice: voxels of 1.0 x 0.5 x 0.5 um
spacing = (1.0, 0.5, 0.5)   # (dz, dy, dx): axis 0 of the mask is z
print(f"stack {stack.shape}, {len(np.unique(stack)) - 1} cells, spacing {spacing} um")

## 1. Voxels longer along z

The resampling alone is `resample_to_isotropic`: shape-based interpolation to cubic voxels of the shortest
spacing.

In [ ]:
import matplotlib.pyplot as plt

resampled = resample_to_isotropic(stack, spacing)
print(f"resampled to {resampled.shape}")

colours = plt.get_cmap("tab20").with_extremes(bad="white")  # the medium in white
figure, axes = plt.subplots(1, 2, figsize=(9, 4.4))
for axis, title, image, aspect in ((axes[0], "the stack (z voxels 2x longer)", stack, 2.0),
                                   (axes[1], "resampled to cubic voxels", resampled, 1.0)):
    section = image[:, image.shape[1] // 2, :]
    axis.imshow(np.ma.masked_equal(section, 0), cmap=colours, interpolation="nearest", aspect=aspect, origin="lower")
    axis.set(title=title, xlabel="x (voxels)", ylabel="z (voxels)")
figure.tight_layout()

`construct_mesh_in_physical_units` resamples and meshes in one call, and returns the mesh in the unit of
`spacing`. `min_distance` counts voxels of the resampled grid, so the mesh has as many points, and costs as much, as
for a mask acquired at the short spacing. The `.rec` file it saves records the frame and the source spacing.

In [ ]:
algorithm = get_default_mesh_reconstruction_algorithm()
points, triangles, labels = algorithm.construct_mesh_in_physical_units(stack, spacing, spacing_unit="um")
print(f"{len(points)} points, {len(triangles)} triangles; extent (z, y, x) in um: "
      f"{np.round(points.max(axis=0) - points.min(axis=0), 1)}")

algorithm.save_to_rec_mesh("anisotropic_stack.rec")
print(load_rec_appendix("anisotropic_stack.rec"))

Meshing the native grid instead gives a mesh in voxel units, squashed along z; `set_pixel_size` scales it
afterwards (column 0, z, by the z size; columns 1 and 2 by the xy size). The mesh then has the right size, but its
surfaces were sampled coarsely along z: measured on 12 simulated tissues whose cells are as small as those of real
tissues, down-sampled along z, against the same tissues meshed at full resolution, meshing the native grid costs
+1.38, +1.97 and +4.10 degrees of contact-angle error at z/xy ratios of 5/3, 2 and 3, against +0.83, +0.71 and
+1.17 for resampling first.

In [ ]:
native = get_default_mesh_reconstruction_algorithm()
native.construct_mesh_from_segmentation_mask(stack)
before = native.last_constructed_mesh[0]
native.set_pixel_size(xy_pixel_size=0.5, z_pixel_size=1.0)
after = native.last_constructed_mesh[0]
print("native grid, voxel units:", np.round(before.max(axis=0) - before.min(axis=0), 1))
print("after set_pixel_size, um: ", np.round(after.max(axis=0) - after.min(axis=0), 1))

## 2. Cells cut by the image box

Cut the foam a third of the way along x, as an image that misses part of the tissue.

In [ ]:
cropped = foam[:, :, foam.shape[2] // 3:]
algorithm = get_default_mesh_reconstruction_algorithm()
points, triangles, labels = algorithm.construct_mesh_from_segmentation_mask(cropped)
print(f"{int(algorithm.crop_faces.sum())} crop faces of {len(triangles)} triangles")
print("cut cells:", algorithm.cut_cells)
print(algorithm.crop_face_report.as_dict())

The mesh itself is the same, bit for bit, as without the marks: the crop faces are a boolean per
triangle. `save_to_rec_mesh` writes them as the last block of triangles and records the block in the file's
appendix; `load_rec_with_crop_faces` reads them back. foambryo keeps them out of its equations and reports the
pressures of the cut cells as `nan`.

In [ ]:
algorithm.save_to_rec_mesh("cropped_foam.rec")
points, triangles, labels, crop_faces = load_rec_with_crop_faces("cropped_foam.rec")
print(f"read back: {int(crop_faces.sum())} crop faces, the last {int(crop_faces.sum())} of {len(triangles)} rows")

If the tissue really ends on an image face (a coverslip), declare that face physical before
reconstructing: its triangles then stay contact with the medium. The faces are named `"axis0_low"`, `"axis0_high"`,
… `"axis2_high"` (`dw3d.IMAGE_FACES`).

In [ ]:
coverslip = get_default_mesh_reconstruction_algorithm()
coverslip.physical_image_faces = ("axis2_low",)   # the cut face, at x = 0 of the cropped mask
coverslip.construct_mesh_from_segmentation_mask(cropped)
print(f"{int(coverslip.crop_faces.sum())} crop faces; cut cells {coverslip.cut_cells}")

To look at the cropped mesh (needs a display):

```python
from dw3d.viewing import plot_cells_polyscope
plot_cells_polyscope(algorithm)
```